# Paper results: every condition in paper.sh

One row per condition, metrics read from each run's `preds/meta.json` — no re-encoding, runs
on CPU in seconds. The condition list is **hard-coded**: the union of the 2026-08-29 run
(the classic/mse/cosent grid, still valid — nothing that trains them changed since) and the
2026-08-31 run (the infonce/siglip families and both ablations). Default metric everywhere:
**Recall@20**; text panels on the left, image on the right.

Sections: 1 health · 2 V ablation · 3 easy ablation · 4 per-strategy comparisons · 5 full table

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from utils.paper_analysis import discover_runs, health_check

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

NOTE = "paper"
MODELS_ROOT = "models"
K_MAIN = 20                      # the paper's headline cutoff
KS = (1, 5, 10, 20, 100)
FIG_DIR = "paper/figs"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Conditions and health

Columns: modality, style, query_kind, V, easy. `V`/`easy` are None/20 for styles that never
see the measured distance. The merge is on all five keys, so the easy-ablation variants of
one (style, V) stay distinct.

In [ ]:
# 2026-08-29 run -- the pair/margin grid; unchanged by anything since.
CONDITIONS_0829 = [
    ("text", "untrained",        "original", None, 20),
    ("text", "untrained",        "synthetic", None, 20),
    ("text", "untrained",        "rephrased", None, 20),
    ("text", "baseline-triplet", "original", None, 20),
    ("text", "baseline-triplet", "synthetic", None, 20),
    ("text", "baseline-triplet", "rephrased", None, 20),
    ("text", "cosent",           "original", None, 20),
    ("text", "cosent",           "synthetic", None, 20),
    ("text", "cosent",           "rephrased", None, 20),
    ("text", "classic-mse",      "original", 40, 20),
    ("text", "classic-mse",      "synthetic", 40, 20),
    ("text", "classic-mse",      "rephrased", 40, 20),
    ("text", "ours-mse",         "original", 40, 20),
    ("text", "ours-mse",         "synthetic", 40, 20),
    ("text", "ours-mse",         "rephrased", 40, 20),
    ("text", "ours-mse-batched", "original", 40, 20),
    ("text", "ours-mse-batched", "synthetic", 40, 20),
    ("text", "ours-mse-batched", "rephrased", 40, 20),
    ("text", "ours-mse",         "synthetic", 20, 20),
    ("text", "ours-mse",         "synthetic", 60, 20),
    ("text", "ours-mse-batched", "synthetic", 20, 20),
    ("text", "ours-mse-batched", "synthetic", 60, 20),
    ("multimodal", "untrained",        "synthetic", None, 20),
    ("multimodal", "untrained",        "rephrased", None, 20),
    ("multimodal", "baseline-triplet", "synthetic", None, 20),
    ("multimodal", "baseline-triplet", "rephrased", None, 20),
    ("multimodal", "cosent",           "synthetic", None, 20),
    ("multimodal", "cosent",           "rephrased", None, 20),
    ("multimodal", "classic-mse",      "synthetic", 40, 20),
    ("multimodal", "classic-mse",      "rephrased", 40, 20),
    ("multimodal", "ours-mse",         "synthetic", 40, 20),
    ("multimodal", "ours-mse",         "rephrased", 40, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 20),
    ("multimodal", "ours-mse-batched", "rephrased", 40, 20),
    ("multimodal", "ours-mse",         "synthetic", 20, 20),
    ("multimodal", "ours-mse",         "synthetic", 60, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 20, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 60, 20),
]

# 2026-08-31 run -- infonce/siglip families, V ablation, easy ablation.
CONDITIONS_0831 = [
    ("text", "infonce",          "original", None, 20),
    ("text", "infonce",          "synthetic", None, 20),
    ("text", "infonce",          "rephrased", None, 20),
    ("text", "infonce-mined",    "original", None, 20),
    ("text", "infonce-mined",    "synthetic", None, 20),
    ("text", "infonce-mined",    "rephrased", None, 20),
    ("text", "siglip-mined",     "original", None, 20),
    ("text", "siglip-mined",     "synthetic", None, 20),
    ("text", "siglip-mined",     "rephrased", None, 20),
    ("text", "ours-infonce",     "original", 40, 20),
    ("text", "ours-infonce",     "synthetic", 40, 20),
    ("text", "ours-infonce",     "rephrased", 40, 20),
    ("text", "ours-siglip",      "original", 40, 20),
    ("text", "ours-siglip",      "synthetic", 40, 20),
    ("text", "ours-siglip",      "rephrased", 40, 20),
    ("text", "ours-infonce",     "synthetic", 20, 20),
    ("text", "ours-infonce",     "synthetic", 60, 20),
    ("text", "ours-siglip",      "synthetic", 20, 20),
    ("text", "ours-siglip",      "synthetic", 60, 20),
    ("text", "classic-mse",      "synthetic", 40, 30),
    ("text", "classic-mse",      "synthetic", 40, 40),
    ("text", "ours-mse",         "synthetic", 40, 30),
    ("text", "ours-mse",         "synthetic", 40, 40),
    ("text", "ours-mse-batched", "synthetic", 40, 30),
    ("text", "ours-mse-batched", "synthetic", 40, 40),
    ("text", "ours-siglip",      "synthetic", 40, 30),
    ("text", "ours-siglip",      "synthetic", 40, 40),
    ("multimodal", "infonce",          "synthetic", None, 20),
    ("multimodal", "infonce",          "rephrased", None, 20),
    ("multimodal", "infonce-mined",    "synthetic", None, 20),
    ("multimodal", "infonce-mined",    "rephrased", None, 20),
    ("multimodal", "siglip-mined",     "synthetic", None, 20),
    ("multimodal", "siglip-mined",     "rephrased", None, 20),
    ("multimodal", "ours-infonce",     "synthetic", 40, 20),
    ("multimodal", "ours-infonce",     "rephrased", 40, 20),
    ("multimodal", "ours-siglip",      "synthetic", 40, 20),
    ("multimodal", "ours-siglip",      "rephrased", 40, 20),
    ("multimodal", "ours-infonce",     "synthetic", 20, 20),
    ("multimodal", "ours-infonce",     "synthetic", 60, 20),
    ("multimodal", "ours-siglip",      "synthetic", 20, 20),
    ("multimodal", "ours-siglip",      "synthetic", 60, 20),
    ("multimodal", "classic-mse",      "synthetic", 40, 30),
    ("multimodal", "classic-mse",      "synthetic", 40, 40),
    ("multimodal", "ours-mse",         "synthetic", 40, 30),
    ("multimodal", "ours-mse",         "synthetic", 40, 40),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 30),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 40),
    ("multimodal", "ours-siglip",      "synthetic", 40, 30),
    ("multimodal", "ours-siglip",      "synthetic", 40, 40),
]

conditions = pd.DataFrame(CONDITIONS_0829 + CONDITIONS_0831,
                          columns=["modality", "style", "query_kind", "V", "easy"])
conditions["V"] = conditions["V"].astype("float64")

runs = discover_runs(MODELS_ROOT, note=NOTE)
runs["easy"] = runs["easy"].fillna(20).astype(int)
matched = conditions.merge(runs, on=["modality", "style", "query_kind", "V", "easy"], how="left")
matched["has_preds"] = matched["has_preds"].fillna(False).astype(bool)

health = health_check(matched)
print(f"{len(conditions)} conditions | {int(health['healthy'].sum())} usable | "
      f"{int((~health['healthy']).sum())} not")
display(health[~health["healthy"]][["modality", "style", "query_kind", "V", "n_queries", "problem"]])
usable = matched[health["healthy"].to_numpy()].reset_index(drop=True)

In [ ]:
def condition_metrics(row):
    meta = json.load(open(os.path.join(row.run_dir, "preds", "meta.json")))
    out = {"modality": row.modality, "style": row.style, "query_kind": row.query_kind,
           "V": row.V, "easy": row.easy, "n_queries": meta["n_queries"]}
    for k in KS:
        out[f"recall@{k}"] = meta["metrics"][f"recall@{k}"]
    return out

results = pd.DataFrame([condition_metrics(r) for r in usable.itertuples()])
results["query_kind"] = pd.Categorical(results["query_kind"],
                                       ["original", "synthetic", "rephrased"], ordered=True)
print(f"{len(results)} conditions loaded")

## 2. V ablation

Recall@20 vs the distance normalizer V, synthetic queries, easy=20. The four styles that were
swept; V=40 is the main-grid value. For ours-infonce, larger V means more target mass on the
hard negative; for ours-siglip it also moves the cross-row target (1 - 20/V).

In [ ]:
SWEPT = ["ours-mse", "ours-mse-batched", "ours-infonce", "ours-siglip"]
sweep = results[results["style"].isin(SWEPT) & (results["query_kind"] == "synthetic")
                & (results["easy"] == 20) & results["V"].notna()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
palette = dict(zip(SWEPT, sns.color_palette("colorblind", len(SWEPT))))
markers = dict(zip(SWEPT, ["o", "D", "s", "^"]))
for ax, modality in zip(axes, ["text", "multimodal"]):
    sub = sweep[sweep["modality"] == modality]
    for style in SWEPT:
        line = sub[sub["style"] == style].sort_values("V")
        ax.plot(line["V"], line[f"recall@{K_MAIN}"], marker=markers[style],
                color=palette[style], label=style)
    ax.set_title(f"{'text' if modality == 'text' else 'image'} (synthetic)")
    ax.set_xlabel("V (distance normalizer)")
    ax.set_ylabel(f"Recall@{K_MAIN}")
    ax.set_xticks([20, 40, 60])
axes[0].legend(fontsize=8)
fig.suptitle("V ablation", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "v_ablation.png"), dpi=150)
plt.show()

display(sweep.pivot_table(index=["modality", "style"], columns="V",
                          values=f"recall@{K_MAIN}").round(4))

## 3. Easy-value ablation

Recall@20 vs the easy-negative distance at V=40, synthetic. easy=20 puts random products
mid-scale (label .5), easy=40 at the scale floor (label 1 -> target 0). ours-infonce is
exempt: its random-negative rows are one-hot at any easy value.

In [ ]:
EASY_STYLES = ["classic-mse", "ours-mse", "ours-mse-batched", "ours-siglip"]
esweep = results[results["style"].isin(EASY_STYLES) & (results["query_kind"] == "synthetic")
                 & (results["V"] == 40)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
palette = dict(zip(EASY_STYLES, sns.color_palette("colorblind", len(EASY_STYLES))))
markers = dict(zip(EASY_STYLES, ["o", "D", "s", "^"]))
for ax, modality in zip(axes, ["text", "multimodal"]):
    sub = esweep[esweep["modality"] == modality]
    for style in EASY_STYLES:
        line = sub[sub["style"] == style].sort_values("easy")
        ax.plot(line["easy"], line[f"recall@{K_MAIN}"], marker=markers[style],
                color=palette[style], label=style)
    ax.set_title(f"{'text' if modality == 'text' else 'image'} (synthetic, V=40)")
    ax.set_xlabel("easy-negative distance")
    ax.set_ylabel(f"Recall@{K_MAIN}")
    ax.set_xticks([20, 30, 40])
axes[0].legend(fontsize=8)
fig.suptitle("Easy-value ablation", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "easy_ablation.png"), dpi=150)
plt.show()

display(esweep.pivot_table(index=["modality", "style"], columns="easy",
                          values=f"recall@{K_MAIN}").round(4))

## 4. Per-strategy comparisons

One figure per strategy family, text left / image right: grouped bars, one group per variant
(plus untrained as a reference group), one bar color per query kind. Groups are ordered by
mean Recall@20, best first. Graded members are their V=40, easy=20 runs. siglip has no
unmined 2-column baseline (never built); cosent's baseline slot is the plain triplet loss,
its nearest ordering-only relative.

In [ ]:
FAMILIES = {
    "infonce": [("infonce", "baseline"), ("infonce-mined", "mined"), ("ours-infonce", "graded")],
    "siglip":  [("siglip-mined", "mined"), ("ours-siglip", "graded")],
    "mse":     [("classic-mse", "baseline"), ("ours-mse", "graded"), ("ours-mse-batched", "batched")],
    "cosent":  [("baseline-triplet", "baseline"), ("cosent", "graded")],
}
QK_ORDER = ["original", "synthetic", "rephrased"]
qk_colour = dict(zip(QK_ORDER, sns.color_palette("colorblind", len(QK_ORDER))))

main = results[(results["easy"] == 20) & (results["V"].isna() | (results["V"] == 40))]

for family, members in FAMILIES.items():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, modality in zip(axes, ["text", "multimodal"]):
        sub = main[main["modality"] == modality]
        kinds = [k for k in QK_ORDER if not sub[sub["query_kind"] == k].empty]
        groups = [(style, f"{style}\n({role})") for style, role in members]
        groups.append(("untrained", "untrained"))
        r20 = {(s, k): sub[(sub["style"] == s) & (sub["query_kind"] == k)][f"recall@{K_MAIN}"].item()
               for s, _ in groups for k in kinds}
        groups.sort(key=lambda g: -sum(r20[g[0], k] for k in kinds) / len(kinds))

        width = 0.8 / len(kinds)
        for j, k in enumerate(kinds):
            xs = [i + (j - (len(kinds) - 1) / 2) * width for i in range(len(groups))]
            ax.bar(xs, [r20[s, k] for s, _ in groups], width=width * 0.95,
                   color=qk_colour[k], label=k)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([label for _, label in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel(f"Recall@{K_MAIN}")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{family} family", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"family_{family}.png"), dpi=150)
    plt.show()

## 5. Full table

Every condition, every cutoff. Also written to paper/figs/all_results.csv.

In [ ]:
table = (results.sort_values(["modality", "query_kind", f"recall@{K_MAIN}"],
                           ascending=[True, True, False])
         .reset_index(drop=True))
table.to_csv(os.path.join(FIG_DIR, "all_results.csv"), index=False)
display(table.round(4))
